# Fundamentals of AI Bootcamp
## Train a tiny neural network: 2 inputs, 2 hidden neurons, 1 output

**Last session:** we chose weights by hand. **Today:** data will change all nine parameters.

Our learning goal is to explain the whole loop: **predict -> measure loss -> compute gradients -> update -> repeat**.
This is **one hidden layer**, plus an output layer (two trainable layers). It is a small, shallow network, not a deep network.

### How to use this notebook
Run from the top. Complete the numbered TODOs one at a time; an unfinished TODO intentionally raises `NotImplementedError`.
Make a prediction in chat before each run. Use the **C01, C02, ...** labels, not the changing execution counters.
A function definition may show no output; the following checkpoint calls it.

The model uses ordinary Python: numbers, lists, tuples, dictionaries, loops and functions. `math.exp` computes an exponential.
`matplotlib` is used only when we need to see the data or training; plotting helpers are supplied, not assessed.
No NumPy, pandas, scikit-learn, PyTorch, GPU, API key, or downloaded dataset is needed.

**Our example is synthetic:** two binary sensor reports, with label 1 when they disagree (XOR).
A real safety system would need much more evidence. Learning all four rows does **not** test generalization.

## C01 | Four examples we can understand completely
Imagine two indicators reporting the same condition. We want to flag disagreement, not diagnose which sensor is broken.
`(x1, x2, target)` is one tuple; `DATA` is a list of those tuples. `for x1, x2, target in DATA` unpacks one row.

**Predict:** what should `(1, 0)` produce? Could we solve this known rule with an `if`? Yes! We use it as a transparent training experiment, not because a neural network is necessary for XOR.

In [ ]:
# C01 - Read the examples
DATA = [
    (0.0, 0.0, 0),
    (0.0, 1.0, 1),
    (1.0, 0.0, 1),
    (1.0, 1.0, 0),
]
for x1, x2, target in DATA:
    print(x1, x2, '->', target)

## C02 | See the problem before solving it
**Predict:** can one straight line put the two 1s on one side and the two 0s on the other?
The plot is supplied. Circles mean agreement; squares mean disagreement. A line cannot separate these four corners.
This is a limitation of a linear boundary, **not of all machine learning**.

In [ ]:
# C02 - Draw the four cases (provided visual helper)
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 5))
for x1, x2, target in DATA:
    marker = 's' if target == 1 else 'o'
    plt.scatter(x1, x2, marker=marker, s=160)
    plt.text(x1 + 0.04, x2 + 0.04, str(target), fontsize=14)
plt.xlim(-0.2, 1.25)
plt.ylim(-0.2, 1.25)
plt.xlabel('Sensor 1: x1')
plt.ylabel('Sensor 2: x2')
plt.title('XOR: can one straight line separate the labels?')
plt.grid(alpha=0.2)
plt.show()

## C03 | From a hard switch to a smooth response
A step function jumps between 0 and 1. Its gradient is zero away from the jump and undefined at the jump.
To train this small network with gradients, we use a **sigmoid**:

$$s(z)=\frac{1}{1+e^{-z}}.$$

`z` is the weighted sum. `s(z)` is the neuron's output. `exp(z)` means $e^z$.
At $z=0$, $e^0=1$, so the output is $1/2$. At negative values it is smaller; at positive values it is larger.
The two equivalent branches below avoid computing an enormous exponential. This is still the same function, not clipping.

**Predict:** order the outputs for `-4`, `0`, and `4` from smallest to largest. We do not round outputs while training.

In [ ]:
# C03 - A smooth activation (provided)
from math import exp

def sigmoid(z):
    if z >= 0:
        return 1.0 / (1.0 + exp(-z))
    exp_z = exp(z)
    return exp_z / (1.0 + exp_z)

for z in [-4, 0, 4]:
    print(z, '->', round(sigmoid(z), 4))
assert sigmoid(0) == 0.5
assert sigmoid(-4) < sigmoid(0) < sigmoid(4)

## C04 | Give every parameter a name
A dictionary is a set of name -> value lookups, just as `came_from` remembered cell -> previous cell in search.
Here `params['w11']` means the weight from input 1 into hidden neuron 1.

| Destination | Weights | Bias |
|---|---|---|
| Hidden neuron 1 | `w11`, `w12` | `b1` |
| Hidden neuron 2 | `w21`, `w22` | `b2` |
| Output neuron | `v1`, `v2` | `bo` |

There are **six weights + three biases = nine parameters**. Input circles hold data, not trainable weights.
These are fixed, unequal starting guesses for a reproducible lesson. They do not encode OR/NAND labels.
Other initializations can learn differently; success is not guaranteed for every start.

In [ ]:
# C04 - Our starting parameters
INITIAL = {
    'w11': 0.8, 'w12': -0.4, 'b1': 0.1,  # hidden neuron 1
    'w21': -0.3, 'w22': 0.9, 'b2': -0.1,  # hidden neuron 2
    'v1': 0.5, 'v2': -0.7, 'bo': 0.0,    # output neuron
}
print('Number of parameters:', len(INITIAL))
print('w11:', INITIAL['w11'])
assert len(INITIAL) == 9

## C05 | The forward pass: three small neurons
Each hidden neuron reads the **same two inputs**, with its **own** weights and bias.
The output neuron reads the **two hidden outputs**, not the raw inputs.

$$h_1=s(w_{11}x_1+w_{12}x_2+b_1),\quad h_2=s(w_{21}x_1+w_{22}x_2+b_2)$$
$$p=s(v_1h_1+v_2h_2+b_o).$$

`h1`, `h2`, and `prediction` are temporary computed values. `params` holds the trainable numbers.
The function returns three values; `h1, h2, prediction = forward(...)` gives each value a name.

**TODO 1:** complete the three weighted sums. **Predict:** will merely calling this function change the weights?

In [ ]:
# C05 - Build the forward pass
def forward(x1, x2, params):
    # TODO 1a: compute z1 and z2 from inputs, weights and biases.
    raise NotImplementedError('TODO 1a: complete both hidden weighted sums')
    h1 = sigmoid(z1)
    h2 = sigmoid(z2)
    # TODO 1b: combine h1 and h2 using v1, v2 and bo.
    raise NotImplementedError('TODO 1b: complete the output weighted sum')
    prediction = sigmoid(zo)
    return h1, h2, prediction

## C06 | Test tiny before training
The first check uses zero weights and biases, so all three sigmoids receive zero.
The last check protects against accidentally updating parameters inside `forward`.
`assert` means: stop here with an error if the statement is false.

In [ ]:
# C06 - Forward-pass checkpoints
zero_params = {}
for name in INITIAL:
    zero_params[name] = 0.0
assert forward(1.0, 0.0, zero_params) == (0.5, 0.5, 0.5)

before = INITIAL.copy()
h1, h2, prediction = forward(1.0, 0.0, INITIAL)
print('h1:', round(h1, 4))
print('h2:', round(h2, 4))
print('prediction:', round(prediction, 4))
assert INITIAL == before
assert 0.70 < h1 < 0.72
assert 0.40 < h2 < 0.41
assert 0.51 < prediction < 0.53
print('Forward-pass checks passed.')

## C07 | One score for all four examples
We reuse last session's mean squared error, deliberately, to keep the loss familiar:

$$L=\frac{1}{n}\sum (p-y)^2.$$

For one prediction `p=0.6` and target `y=1`, error is `-0.4` and squared error is `0.16`.
MSE is valid for this demonstration; binary cross-entropy is a common alternative for classification, not required today.
**TODO 2:** accumulate each squared error and take the mean. **Predict:** what is the MSE of predicting 0.5 on every XOR row?

In [ ]:
# C07 - Measure the loss
def mean_loss(data, params):
    if len(data) == 0:
        raise ValueError('Need at least one labeled example.')
    total = 0.0
    for x1, x2, target in data:
        h1, h2, prediction = forward(x1, x2, params)
        # TODO 2a: add the squared error for this example.
        raise NotImplementedError('TODO 2a: accumulate squared error')
    # TODO 2b: return the average, not the sum.
    raise NotImplementedError('TODO 2b: return the mean loss')

assert mean_loss(DATA, zero_params) == 0.25
print('Initial MSE:', round(mean_loss(DATA, INITIAL), 6))

## C08 | Follow one influence: output weight -> prediction -> loss
For one example, $\ell=(p-y)^2$. The sigmoid derivative is $s'(z)=s(z)(1-s(z))$.
We multiply local sensitivities along a path (the chain rule):

$$d_o=\frac{\partial\ell}{\partial z_o}=2(p-y)p(1-p),\qquad
\frac{\partial\ell}{\partial v_1}=d_o h_1.$$

`d_out` is how the loss changes if the **output weighted sum** increases slightly.
`dv1` is how the loss changes if output weight `v1` increases slightly.
The gradient points uphill; we later **subtract** a small multiple to go downhill.

**TODO 3:** compute `d_out` and `dv1`. First predict their signs for the example below.

In [ ]:
# C08 - One output-weight gradient
x1, x2, target = 1.0, 0.0, 1
h1, h2, prediction = forward(x1, x2, INITIAL)
error = prediction - target
# TODO 3a: multiply the loss slope and sigmoid slope.
raise NotImplementedError('TODO 3a: compute d_out')
# TODO 3b: how strongly does v1 affect the weighted sum?
raise NotImplementedError('TODO 3b: compute dv1')
print('error:', round(error, 6))
print('d_out:', round(d_out, 6))
print('dv1:', round(dv1, 6))
assert d_out < 0 and dv1 < 0

## C09 | The hidden neuron is not given its own correct answer
Its influence travels through the output weight:

$$d_{h1}=d_o v_1 h_1(1-h_1),\qquad
\frac{\partial\ell}{\partial w_{11}}=d_{h1}x_1.$$

`d_hidden1` is the loss sensitivity to the weighted sum `z1`, not the activation `h1` itself.
The corresponding path for hidden neuron 2 uses `v2` and `h2`.

**TODO 4:** compute `d_hidden1` and `dw11`. **Predict:** why would the gradient of `w12` be zero for this example with `x2=0`?

In [ ]:
# C09 - Pass sensitivity back to a hidden weight
# TODO 4a: multiply the three influences on the hidden path.
raise NotImplementedError('TODO 4a: compute d_hidden1')
# TODO 4b: include the input carried by this connection.
raise NotImplementedError('TODO 4b: compute dw11')
dw12 = d_hidden1 * x2
print('d_hidden1:', round(d_hidden1, 6))
print('dw11:', round(dw11, 6))
print('dw12:', round(dw12, 6))
assert dw12 == 0.0

## C10 | Listen to all four examples before updating
`gradients` repeats the same calculation on each row, adds the nine contributions, then divides by the number of rows.
This gives the gradient of the **mean** loss. No weights change inside this function.

**TODO 5:** reuse the output and hidden sensitivity formulas. The repetitive bookkeeping is supplied.
**Important:** compute all gradients with the old parameters. Do not update `v1` before computing hidden gradients.

In [ ]:
# C10 - Compute all nine mean gradients
def gradients(data, params):
    if len(data) == 0:
        raise ValueError('Need at least one labeled example.')
    g = {}
    for name in params:
        g[name] = 0.0

    for x1, x2, target in data:
        h1, h2, prediction = forward(x1, x2, params)
        # TODO 5: reuse C08-C09; include the second hidden neuron.
        raise NotImplementedError('TODO 5: compute all three sensitivities')

        # Output layer: incoming activation for weights, 1 for the bias.
        g['v1'] += d_out * h1
        g['v2'] += d_out * h2
        g['bo'] += d_out
        # Hidden neuron 1: incoming raw input for each weight.
        g['w11'] += d_hidden1 * x1
        g['w12'] += d_hidden1 * x2
        g['b1'] += d_hidden1
        # Hidden neuron 2: same pattern, different values.
        g['w21'] += d_hidden2 * x1
        g['w22'] += d_hidden2 * x2
        g['b2'] += d_hidden2

    for name in g:
        g[name] /= len(data)
    return g

one_row = gradients([(1.0, 0.0, 1)], INITIAL)
assert abs(one_row['v1'] - dv1) < 1e-12
assert abs(one_row['w11'] - dw11) < 1e-12
assert set(gradients(DATA, INITIAL)) == set(INITIAL)
print('All nine gradient entries are present.')

## C11 | This is the training loop
An **epoch** here means one pass over the entire four-row dataset. We use one full-batch update per epoch.
`learning_rate` is our step size, not a parameter learned from the examples.
`history` stores losses. `snapshots` stores occasional copies of the weights so we can replay learning later.

**TODO 6:** get the gradients, then update every parameter with `old value - learning_rate * gradient`.
`initial.copy()` means each experiment starts from the same numbers. The original dictionary must not be changed.

In [ ]:
# C11 - Update, then repeat
def train(data, initial, learning_rate=1.0, epochs=8000, record_every=100):
    if learning_rate <= 0 or epochs < 0 or record_every < 1:
        raise ValueError('Use a positive learning rate and record interval; epochs >= 0.')
    params = initial.copy()
    history = [mean_loss(data, params)]
    snapshots = [(0, params.copy())]
    for epoch in range(1, epochs + 1):
        # TODO 6a: get all gradients from the current parameters.
        raise NotImplementedError('TODO 6a: compute g')
        for name in params:
            # TODO 6b: apply gradient descent to this parameter.
            raise NotImplementedError('TODO 6b: subtract step size times gradient')
        history.append(mean_loss(data, params))
        if epoch % record_every == 0 or epoch == epochs:
            snapshots.append((epoch, params.copy()))
    return params, history, snapshots

unchanged = INITIAL.copy()
one_step, tiny_history, tiny_frames = train(DATA, INITIAL, epochs=1)
assert INITIAL == unchanged
assert len(tiny_history) == 2
assert tiny_history[-1] < tiny_history[0]
print('One update lowers this batch loss:', tiny_history)

## C12 | Train your network
**Predict before running:** will the first and final losses match? Which dictionary entries should change?
A plateau is not proof of a bug. Run with the supplied initialization first before experimenting.
This fixed example should reach MSE below 0.005; other initializations need not.

In [ ]:
# C12 - Run learning and inspect the evidence
learned, history, snapshots = train(DATA, INITIAL, learning_rate=1.0, epochs=8000)
print('Starting loss:', round(history[0], 6))
print('Final loss:   ', round(history[-1], 6))
for name in INITIAL:
    print(name, round(INITIAL[name], 3), '->', round(learned[name], 3))
assert history[-1] < 0.005
assert len(history) == 8001
assert learned != INITIAL

## C13 | Turn a score into a displayed decision
Training used continuous scores. Only now do we apply a threshold of 0.5.
This bounded score is not automatically a calibrated probability or confidence guarantee.

**TODO 7:** output 1 when the score is at least 0.5, otherwise output 0.
**Discuss:** if all four rows are correct, have we tested on any new examples? No.

In [ ]:
# C13 - Evaluate the four known rows
correct = 0
for x1, x2, target in DATA:
    h1, h2, prediction = forward(x1, x2, learned)
    # TODO 7: make the reporting decision, not a training activation.
    raise NotImplementedError('TODO 7: threshold the score')
    print((x1, x2), 'score', round(prediction, 3), 'decision', decision, 'target', target)
    if decision == target:
        correct += 1
print('Truth-table accuracy:', correct, '/', len(DATA))
assert correct == 4

## C14 | Make learning visible
The first plot follows MSE over epochs. The second plots the same four examples in the space of hidden outputs `(h1,h2)`.
Those outputs are learned features, not labels we wrote for the hidden neurons.
**Predict:** can the output neuron now separate these transformed points with a straight boundary in hidden space?

In [ ]:
# C14 - Loss curve and learned features (provided)
plt.figure(figsize=(7, 4))
plt.plot(range(len(history)), history)
plt.xlabel('Epoch (one full-batch update)')
plt.ylabel('Mean squared error')
plt.title('Same data and code; changing weights and biases')
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(5, 5))
for x1, x2, target in DATA:
    h1, h2, prediction = forward(x1, x2, learned)
    plt.scatter(h1, h2, marker='s' if target == 1 else 'o', s=150)
    plt.text(h1 + 0.02, h2 + 0.02, str((int(x1), int(x2))), fontsize=10)
# The output decision boundary satisfies v1*h1 + v2*h2 + bo = 0.
if abs(learned['v2']) > 1e-12:
    horizontal = [-0.1, 1.1]
    vertical = []
    for value in horizontal:
        vertical.append(-(learned['v1'] * value + learned['bo']) / learned['v2'])
    plt.plot(horizontal, vertical, '--', label='output boundary')
plt.xlim(-0.1, 1.15)
plt.ylim(-0.1, 1.15)
plt.xlabel('Hidden output h1')
plt.ylabel('Hidden output h2')
plt.title('Learned features: the four inputs after transformation')
plt.grid(alpha=0.2)
plt.show()

## C15 | Replay the snapshots (optional, after the core works)
This supplied cell creates a self-contained play/pause scrubber from **your actual training snapshots**.
It uses `IPython.display.HTML` only to display the animation. Some notebook viewers disable JavaScript; use the separate `media/Training_Lab.html` or the MP4 in that case.
The four bars are the model scores on the four labeled corners. The epoch label jumps because we saved every 100th update.

In [ ]:
# C15 - Replay real training states (provided)
import json
from IPython.display import HTML, display

replay = []
for epoch, saved_params in snapshots:
    scores = []
    for x1, x2, target in DATA:
        scores.append(forward(x1, x2, saved_params)[2])
    replay.append({'epoch': epoch, 'scores': scores, 'loss': history[epoch]})

replay_html = r"""
<div class="nn-replay" style="font-family:sans-serif;max-width:760px;padding:18px;border:1px solid #bbb">
 <strong>Our network learning: four known examples</strong><br>
 <button class="play">Play / Pause</button>
 <input class="slider" type="range" min="0" value="0" style="width:55%">
 <span class="label"></span>
 <div class="bars"></div>
 <small>These are training cases, not held-out test examples.</small>
</div>
<script>
(function(){
 const script=document.currentScript, root=script.previousElementSibling;
 const data=__DATA__, slider=root.querySelector('.slider'), label=root.querySelector('.label');
 slider.max=data.length-1;
 let timer=null;
 function draw(){
   const s=data[Number(slider.value)];
   label.textContent='Epoch '+s.epoch+' | MSE '+s.loss.toFixed(5);
   root.querySelector('.bars').innerHTML=s.scores.map((p,i)=>
     '<div style="margin:12px 0">'+['00 -> 0','01 -> 1','10 -> 1','11 -> 0'][i]+
     ' : '+p.toFixed(3)+'<div style="background:#ddd;height:14px"><div style="background:#2a9d8f;height:14px;width:'+100*p+'%"></div></div></div>').join('');
 }
 slider.oninput=draw;
 root.querySelector('.play').onclick=function(){
   if(timer){clearInterval(timer);timer=null;return;}
   if(Number(slider.value)>=data.length-1)slider.value=0;
   timer=setInterval(function(){
     slider.value=Number(slider.value)+1;draw();
     if(Number(slider.value)>=data.length-1){clearInterval(timer);timer=null;}
   },120);
 };
 draw();
})();
</script>"""
display(HTML(replay_html.replace('__DATA__', json.dumps(replay))))

## C16 | Break it safely: initialization and step size
Start every comparison from a fresh copy. Change **one** thing at a time and keep the same epoch budget.
For balanced XOR, all-zero initialization has zero full-batch gradient and stays at 0.5. That is a model-training failure, not an infinite loop.

**Predict:** what changes with learning rates 0.01 and 1.0 after 2,000 epochs? Smaller is not always better; larger is not always better either.

In [ ]:
# C16 - Controlled failure experiments
zero_learned, zero_history, _ = train(DATA, zero_params, epochs=100)
print('All zeros: first/final MSE', zero_history[0], zero_history[-1])
assert zero_learned == zero_params
assert zero_history[-1] == 0.25

for rate in [0.01, 1.0]:
    trial, trial_history, _ = train(DATA, INITIAL, learning_rate=rate, epochs=2000)
    print('learning rate', rate, 'final MSE', round(trial_history[-1], 6))

## C17 | Optional mathematical check: does our gradient match a tiny nudge?
For each parameter, temporarily increase and decrease it by a tiny amount, holding all others fixed.
The central-difference slope should nearly match the analytic gradient. This is a debugging tool, not our main training algorithm.

$$\frac{L(\theta+\varepsilon)-L(\theta-\varepsilon)}{2\varepsilon}\approx\frac{\partial L}{\partial\theta}.$$

**Predict:** why must `plus` and `minus` be separate copies?

In [ ]:
# C17 - Optional gradient check (provided)
epsilon = 1e-5
analytic = gradients(DATA, INITIAL)
for name in INITIAL:
    plus = INITIAL.copy()
    minus = INITIAL.copy()
    plus[name] += epsilon
    minus[name] -= epsilon
    estimated = (mean_loss(DATA, plus) - mean_loss(DATA, minus)) / (2 * epsilon)
    assert abs(analytic[name] - estimated) < 1e-7, name
print('All nine gradients agree with small numerical nudges.')

## Exit ticket | Explain rather than memorize
1. Point to a weight, a bias and an activation. Which persist between examples? Which are recomputed?
2. Why did we replace the step activation? Why do we still use a threshold in C13?
3. What does backpropagation compute? What does gradient descent do with its result?
4. How many hidden layers does our network have? Is it a deep network?
5. We classified all four binary inputs correctly. What claim is justified, and what claim is not?
6. Suggest one controlled experiment. Name the change, the fixed conditions, the metric and the expected failure.

**Project bridge:** open `Week04_Project_Starter_Student.ipynb` for simulated, non-binary sensor readings and a genuine train/validation/test split.
Do not confuse those new simulated examples with measurements from a real device.

### Reference trail (optional)
This lesson continues the instructor's Week03 slides: XOR, OR/NAND/AND, hidden layers and the promise to train the weights.
The equations and numerical examples here are derived specifically for this nine-parameter model.
Further visual explanations: 3Blue1Brown, *Gradient descent, how neural networks learn* and *Backpropagation calculus*.
Instructor reference: Goodfellow, Bengio and Courville, *Deep Learning*, Chapter 6 (feedforward networks and backpropagation).